<a href="https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Method choice

I will use a Random Forest classifier for this provisional modeling task.

My target is a proxy based on observed performance direction: whether a page has a declining trend.

A Random Forest is suitable for this exploratory baseline because the dataset contains a mixture of numeric and categorical page-level signals, and the relationships between search performance, engagement, content age, and freshness may not be captured well by a single fixed rule.

The model will be compared with my Week-4 baseline using the same held-out test data and the same ranking metric, Precision@50.

This model is decision-support. A high score means a page may deserve review; it does not prove that refreshing the page will improve performance.

In [5]:
# ==========================================
# SECTION 1 — LOAD DATA AND METHOD CHECK
# ==========================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# Load data
url = "https://raw.githubusercontent.com/Rimshakalhoro/flyrank-ml-internship-rimsha/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nTarget source:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nMethod:")
print("Random Forest Classifier")


Rows: 30000
Columns: 44

Target source:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Method:
Random Forest Classifier


### Split design

I use a grouped split based on `client_id`.

The reason is that multiple content pages may belong to the same client. If pages from the same client appear in both training and test data, the evaluation may be overly optimistic because client-specific patterns could appear on both sides.

The model is therefore trained on pages from one group of clients and evaluated on different clients.

The split is not time-aware because the starter dataframe does not provide explicit calendar dates for each row. Therefore, this evaluation should be interpreted as a grouped generalization check rather than a simulation of future prediction.

In [6]:
# ==========================================
# SECTION 2 — CREATE TARGET AND GROUPED SPLIT
# ==========================================

from sklearn.model_selection import GroupShuffleSplit

model_df = df.copy()

# Create binary target
# 1 = observed declining trend
# 0 = other observed trend direction
model_df["target_declining"] = (
    model_df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Target distribution:")
print(model_df["target_declining"].value_counts())

# Grouped split by client
groups = model_df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(model_df, groups=groups)
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("\nTraining rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nUnique clients in training:")
print(train_df["client_id"].nunique())

print("\nUnique clients in testing:")
print(test_df["client_id"].nunique())

# Check overlap
overlap = set(train_df["client_id"]).intersection(
    set(test_df["client_id"])
)

print("\nClient overlap between train and test:", len(overlap))


Target distribution:
target_declining
1    16262
0    13738
Name: count, dtype: int64

Training rows: 23837
Testing rows: 6163

Unique clients in training:
25

Unique clients in testing:
7

Client overlap between train and test: 0


### Train and compare with my baseline

I train the model on the training portion of the grouped split and evaluate it on the same held-out clients used for the baseline comparison.

The target is the observed declining-trend proxy. The model does not use `trend_direction` as a feature because that would directly reveal the target.

I compare the learned model with the Week-4 rule-based baseline using Precision@50. This focuses the evaluation on the quality of the highest-priority recommendations.

The comparison is exploratory and should not be interpreted as proof that one approach will always perform better in production.

In [7]:
# ==========================================
# SECTION 3 — TRAIN MODEL AND COMPARE
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------
# DEFINE COLUMNS THAT MUST NOT BE FEATURES
# ------------------------------------------

exclude_columns = [
    "content_id",          # identifier
    "client_id",           # grouping identifier
    "trend_direction",     # source of target
    "target_declining",    # target itself
    "trend_pct"            # closely related to target/trend outcome
]

feature_columns = [
    col for col in model_df.columns
    if col not in exclude_columns
]

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["target_declining"]
y_test = test_df["target_declining"]

# ------------------------------------------
# IDENTIFY NUMERIC AND CATEGORICAL FEATURES
# ------------------------------------------

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Number of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

# ------------------------------------------
# PREPROCESSING
# ------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# ------------------------------------------
# MODEL
# ------------------------------------------

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

# Train
pipeline.fit(X_train, y_train)

# Predict probability of declining trend
test_df = test_df.copy()

test_df["model_score"] = pipeline.predict_proba(
    X_test
)[:, 1]

# ------------------------------------------
# PRECISION@50 FUNCTION
# ------------------------------------------

def precision_at_k(data, score_column, target_column, k=50):

    top_k = (
        data
        .sort_values(score_column, ascending=False)
        .head(k)
    )

    return top_k[target_column].mean()

model_precision_50 = precision_at_k(
    test_df,
    "model_score",
    "target_declining",
    k=50
)

print("MODEL PRECISION@50:")
print(round(model_precision_50, 4))

# ------------------------------------------
# BUILD WEEK-4 STYLE BASELINE
# ON THE SAME TEST DATA
# ------------------------------------------

baseline_test = test_df.copy()

baseline_test["baseline_score"] = 0

# Rule 1 — declining trend
# NOTE: We do NOT use trend_direction here because
# it is the target and would be leakage.
# The baseline below uses only observable input signals.

impression_threshold = train_df[
    "impressions_90d"
].quantile(0.75)

engagement_threshold = train_df[
    "engagement_rate"
].quantile(0.25)

stale_threshold = train_df[
    "days_since_last_update"
].quantile(0.75)

# High visibility
baseline_test.loc[
    baseline_test["impressions_90d"] >= impression_threshold,
    "baseline_score"
] += 2

# Low engagement
baseline_test.loc[
    baseline_test["engagement_rate"] <= engagement_threshold,
    "baseline_score"
] += 1

# Stale content
baseline_test.loc[
    baseline_test["days_since_last_update"] >= stale_threshold,
    "baseline_score"
] += 1

baseline_precision_50 = precision_at_k(
    baseline_test,
    "baseline_score",
    "target_declining",
    k=50
)

# ------------------------------------------
# COMPARISON TABLE
# ------------------------------------------

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline Rule",
        "Random Forest Model"
    ],
    "Precision@50": [
        baseline_precision_50,
        model_precision_50
    ]
})

comparison["Precision@50"] = comparison[
    "Precision@50"
].round(4)

print("\nCOMPARISON:")

comparison


Number of numeric features: 29
Number of categorical features: 11
MODEL PRECISION@50:
0.92

COMPARISON:


,Method,Precision@50
0,Week-4 Baseline Rule,0.46
1,Random Forest Model,0.92


### Errors and interpretation

I examine pages where the model assigns a high score but the observed target is not declining, and pages where the observed target is declining but the model assigns a low score.

A high-scoring false positive may occur when a page has signals that look similar to declining pages but the observed trend did not decline.

A low-scoring false negative may occur when a declining page has unusual characteristics that are not well represented in the training data.

The model's feature importance can provide clues about which observable signals it relies on. Feature importance should be interpreted as model behavior, not as proof of causation or proof that a feature is a Google ranking factor.

This analysis is intended to improve decision-support and identify limitations rather than claim that the model has discovered a causal explanation.

In [8]:
# ==========================================
# SECTION 4 — ERROR ANALYSIS
# ==========================================

# Create predicted class using 0.50 threshold
test_df["predicted_declining"] = (
    test_df["model_score"] >= 0.50
).astype(int)

# ------------------------------------------
# FALSE POSITIVES
# Predicted declining, but target was not declining
# ------------------------------------------

false_positives = test_df[
    (test_df["predicted_declining"] == 1)
    &
    (test_df["target_declining"] == 0)
].sort_values(
    "model_score",
    ascending=False
)

print("FALSE POSITIVES:")
print("Count:", len(false_positives))

false_positives[
    [
        "content_id",
        "model_score",
        "target_declining",
        "impressions_90d",
        "engagement_rate",
        "days_since_last_update"
    ]
].head(10)


FALSE POSITIVES:
Count: 1012


,content_id,model_score,target_declining,impressions_90d,engagement_rate,days_since_last_update
20736,content_41baf0722ad9,0.855798,0,3115,0.00,104
2357,content_8f1409b2674e,0.849530,0,209,0.00,104
12332,content_4d9f36001f06,0.849290,0,3369,33.33,104
11061,content_0b47dae0c7f9,0.849283,0,1191,0.00,103
28718,content_ef6e7d7cfe15,0.830504,0,264,0.00,104
1517,content_816d77e36e14,0.826981,0,208,0.00,104
4050,content_500bd3907331,0.822822,0,4037,0.00,104
5399,content_6677fd6c4ea5,0.821456,0,1152,0.00,104
10080,content_35d63627bf3e,0.816633,0,1525,0.00,103
29456,content_b46c62b14582,0.815378,0,6240,0.00,103


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.